<a href="https://colab.research.google.com/github/kinshukpcs24-ux/BIS_Lab/blob/main/1BM24CS137_Population_Growth_Particle_Swarm_Optimization/1BM24CS137_Population_Growth_Particle_Swarm_Optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import math
import random

# Target population data we want to model and fit
# Let's say we have actual observed population values over 10 generations/years
target_population = [100, 150, 220, 176, 343, 534, 570, 610, 630, 640]
time_steps = list(range(len(target_population)))
initial_pop = target_population[0]

def simulate_population(r, K, steps=10):
    """Simulates logistic population growth: P_t+1 = P_t + r * P_t * (1 - P_t / K)"""
    pop = [initial_pop]
    for _ in range(steps - 1):
        p_current = pop[-1]
        p_next = p_current + r * p_current * (1 - p_current / K)
        # Keep population non-negative
        pop.append(max(0.0, p_next))
    return pop

def fitness(params):
    """Calculates fitness based on mean squared error. We minimize error, so fitness is -MSE."""
    r, K = params
    # Boundary constraints
    if r <= 0 or K <= 0:
        return -9e9
    simulated = simulate_population(r, K, len(target_population))
    mse = sum((sim - tar) ** 2 for sim, tar in zip(simulated, target_population)) / len(target_population)
    return -mse

def main():
    num_particles = 50
    iterations = 1000

    w = 0.7   # Inertia weight
    c1 = 1.5  # Cognitive coefficient
    c2 = 0.3  # Social coefficient

    # Parameter bounds: r in [0.01, 1.0], K in [500, 1000]
    bounds = [(0.01, 1.0), (500.0, 1000.0)]

    # Initialize particle positions and velocities
    positions = []
    velocities = []
    for _ in range(num_particles):
        pos = [
            random.uniform(bounds[0][0], bounds[0][1]),
            random.uniform(bounds[1][0], bounds[1][1])
        ]
        vel = [
            random.uniform(-0.1, 0.1),
            random.uniform(-10, 10)
        ]
        positions.append(pos)
        velocities.append(vel)

    # Initialize personal bests
    p_best = list(positions)
    p_best_fitness = [fitness(p) for p in positions]

    # Initialize global best
    g_best_idx = p_best_fitness.index(max(p_best_fitness))
    g_best = list(p_best[g_best_idx])
    g_best_fit = p_best_fitness[g_best_idx]

    # PSO Optimization Loop
    for _ in range(iterations):
        for i in range(num_particles):
            r1, r2 = random.random(), random.random()

            # Update velocities and positions for each dimension
            for d in range(2):
                velocities[i][d] = (w * velocities[i][d] +
                                    c1 * r1 * (p_best[i][d] - positions[i][d]) +
                                    c2 * r2 * (g_best[d] - positions[i][d]))
                positions[i][d] += velocities[i][d]

                # Keep within bounds
                positions[i][d] = max(bounds[d][0], min(positions[i][d], bounds[d][1]))

            # Update personal and global bests
            current_fitness = fitness(positions[i])
            if current_fitness > p_best_fitness[i]:
                p_best[i] = list(positions[i])
                p_best_fitness[i] = current_fitness

                if current_fitness > g_best_fit:
                    g_best = list(positions[i])
                    g_best_fit = current_fitness

    # Results
    best_r, best_K = g_best
    best_sim = simulate_population(best_r, best_K, len(target_population))

    print(f"Results after {iterations} PSO iterations:")
    print(f"Optimized Growth Rate (r): {best_r:.4f}")
    print(f"Optimized Carrying Capacity (K): {best_K:.2f}")
    print(f"Best Fitness (negative MSE): {g_best_fit:.4f}")

if __name__ == '__main__':
    main()

Results after 1000 PSO iterations:
Optimized Growth Rate (r): 0.5422
Optimized Carrying Capacity (K): 677.48
Best Fitness (negative MSE): -1905.5811
